# AGNI — Dynamic Multi-Hop Geopolitical Risk Graph Topology
### AstraX Sovereign Research Intelligence Platform

This notebook constructs and validates the **Dynamic Risk Graph Topology** modeling how physical chokepoints transmit shocks to global commodities and financial assets.

**Causal Chain:**
$$\text{Country} \longrightarrow \text{Chokepoint} \longrightarrow \text{Commodity} \longrightarrow \text{Financial Asset} \longrightarrow \text{Macro Volatility}$$


In [ ]:
# 1. Environment & Dependencies Installation
!pip install -q networkx torch scipy

In [ ]:
# 2. Graph Construction with NetworkX
import networkx as nx
import json
from datetime import datetime

G = nx.DiGraph()

# Add Nodes with categorical types
nodes = [
    ("IRN", {"type": "country", "name": "Iran"}),
    ("YEM", {"type": "country", "name": "Yemen"}),
    ("TWN", {"type": "country", "name": "Taiwan"}),
    ("strait_of_hormuz", {"type": "chokepoint", "name": "Strait of Hormuz", "trade_share": 0.21}),
    ("bab_el_mandeb", {"type": "chokepoint", "name": "Bab el-Mandeb", "trade_share": 0.12}),
    ("taiwan_strait", {"type": "chokepoint", "name": "Taiwan Strait", "trade_share": 0.48}),
    ("BRENT_CRUDE", {"type": "commodity", "name": "Brent Crude Oil"}),
    ("CONTAINER_SCFI", {"type": "commodity", "name": "Container Freight Index"}),
    ("SPX", {"type": "financial_asset", "name": "S&P 500 Index"}),
    ("VIX", {"type": "volatility", "name": "CBOE Volatility Index"}),
]
G.add_nodes_from(nodes)

# Add Directed Causal Edges with Transmission Elasticities
edges = [
    ("IRN", "strait_of_hormuz", {"link_type": "geopolitical_shock", "elasticity": 1.45}),
    ("strait_of_hormuz", "BRENT_CRUDE", {"link_type": "supply_disruption", "elasticity": 1.80}),
    ("BRENT_CRUDE", "SPX", {"link_type": "price_transmission", "elasticity": -0.32}),
    ("BRENT_CRUDE", "VIX", {"link_type": "volatility_contagion", "elasticity": 0.65}),
    ("YEM", "bab_el_mandeb", {"link_type": "geopolitical_shock", "elasticity": 1.25}),
    ("bab_el_mandeb", "CONTAINER_SCFI", {"link_type": "supply_disruption", "elasticity": 2.10}),
    ("CONTAINER_SCFI", "SPX", {"link_type": "price_transmission", "elasticity": -0.28}),
    ("TWN", "taiwan_strait", {"link_type": "geopolitical_shock", "elasticity": 1.60}),
    ("taiwan_strait", "SPX", {"link_type": "supply_disruption", "elasticity": -0.55}),
]
G.add_edges_from(edges)

print(f"[AGNI] Risk Graph Initialized: {G.number_of_nodes()} nodes, {G.number_of_edges()} causal links.")

In [ ]:
# 3. Topological Centrality Metrics
betweenness = nx.betweenness_centrality(G)
in_degree = dict(G.in_degree())
out_degree = dict(G.out_degree())

print("Topological Betweenness Centralities:")
for node, bc in sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {node:<20}: {bc:.4f}")

In [ ]:
# 4. Multi-Hop Shortest Path Tracing from Shocks to Portfolio Assets
def trace_transmission_path(source_country, target_asset):
    try:
        path = nx.shortest_path(G, source=source_country, target=target_asset)
        cum_elasticity = 1.0
        for u, v in zip(path[:-1], path[1:]):
            cum_elasticity *= G[u][v]["elasticity"]
        return path, cum_elasticity
    except nx.NetworkXNoPath:
        return [], 0.0

p1, e1 = trace_transmission_path("IRN", "VIX")
print(f"Transmission Path (IRN -> VIX): {' -> '.join(p1)} (Cumulative Beta: {e1:.2f})")

p2, e2 = trace_transmission_path("YEM", "SPX")
print(f"Transmission Path (YEM -> SPX): {' -> '.join(p2)} (Cumulative Beta: {e2:.2f})")

In [ ]:
# 5. Export Graph Artifacts for Production Inference
graph_export = {
    "nodes": [{"id": n, **G.nodes[n]} for n in G.nodes],
    "edges": [{"from": u, "to": v, **G[u][v]} for u, v in G.edges],
    "centralities": betweenness,
    "exported_at": datetime.utcnow().isoformat()
}
with open("dynamic_risk_graph.json", "w") as f:
    json.dump(graph_export, f, indent=2)
print("[AGNI] Graph artifact saved: dynamic_risk_graph.json")